## 01 Data Schema and Cleaning

Load, explore, and clean raw H&M datasets: articles, customers, transactions

Output:

- cleaned articles data
- cleaned customers data
- cleaned transactions data (50,000 samples)
- merged dataset


#### Setup


In [ ]:
import os
import pandas as pd
import numpy as np

RAW_DIR = "../data/raw/hm"
PROCESSED_DIR = "../data/processed/hm"
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [15]:
# Load raw files
articles = pd.read_csv(os.path.join(RAW_DIR, "articles.csv"))
customers = pd.read_csv(os.path.join(RAW_DIR, "customers.csv"))
transactions = pd.read_csv(os.path.join(RAW_DIR, "transactions.csv"))

In [ ]:
# Check files size
print(articles.shape, customers.shape, transactions.shape)

(105542, 25) (1371980, 7) (31788324, 5)


#### Clean product catalog


In [29]:
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [ ]:

# Rename columns
articles_clean = articles.rename(columns={
    "prod_name": "product_name",
    "product_type_name": "product_type",
    "product_group_name": "product_group",
    "graphical_appearance_name": "graphical_appearance",
    "colour_group_name": "color_group",
    "perceived_colour_master_name": "perceived_color",
    "department_name": "department",
    "index_name": "index",
    "index_group_name": "index_group",
    "section_name": "section",
    "garment_group_name": "garment_group",
    "detail_desc": "description"
})

In [17]:
# Add image paths
def get_image_path(article_id):
    article_id = str(article_id).zfill(10)
    folder = article_id[:3]
    return os.path.join(RAW_DIR, "images", folder, f"{article_id}.jpg")

articles_clean["image_path"] = articles_clean["article_id"].apply(get_image_path)
articles_clean["image_exists"] = articles_clean["image_path"].apply(os.path.exists)

In [ ]:
# Check image coverage
print("Image coverage:", articles_clean["image_exists"].mean())

Image coverage: 0.9958120937636201


#### Clean customers


In [30]:
customers.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [ ]:
# Check nan values
customers.isna().sum()

# customers["FN"].unique()
# customers["Active"].unique()
# customers["club_member_status"].unique()
# customers["fashion_news_frequency"].unique()
# customers["age"].unique()

customer_id                    0
FN                        895050
Active                    907576
club_member_status          6062
fashion_news_frequency     16011
age                        15861
postal_code                    0
dtype: int64

In [ ]:
# Rename columns and handle missing values
customers_clean = customers.rename(columns={
    "FN": "fashion_news_binary",
    "Active": "is_active"
})

customers_clean["fashion_news_binary"] = customers_clean["fashion_news_binary"].fillna(0)
customers_clean["is_active"] = customers_clean["is_active"].fillna(0)
customers_clean["club_member_status"] = customers_clean["club_member_status"].fillna("UNKNOWN")
customers_clean["fashion_news_frequency"] = customers_clean["fashion_news_frequency"].fillna("UNKNOWN")
customers_clean["age"] = customers_clean["age"].fillna(customers_clean["age"].median())

In [ ]:
# Add age groups
def assign_age_group(age):
    if age < 25:
        return "Gen Z"
    elif age < 35:
        return "Young Adult"
    elif age < 50:
        return "Adult"
    else:
        return "Mature"

customers_clean["age_group"] = customers_clean["age"].apply(assign_age_group)

#### Clean transactions


In [44]:
transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [ ]:

# Rename columns and convert date
transactions_clean = transactions.rename(columns={
    "t_dat": "transaction_date"
})

transactions_clean["transaction_date"] = pd.to_datetime(transactions_clean["transaction_date"])

# Sample transactions for speed
transactions_sample = transactions_clean.sample(500000, random_state=42) 

#### Create product popularity features


In [ ]:

articles_stats = (
    transactions_sample
    .groupby("article_id")
    .agg(
        product_purchase_count=("customer_id", "count"),
        unique_customer_count=("customer_id", "nunique"),
        avg_selling_price=("price", "mean") # price has been normalized, we treat it as relative pricing signal
    )
    .reset_index()
)

articles_clean = articles_clean.merge(articles_stats, on="article_id", how="left")

articles_clean[["product_purchase_count", "unique_customer_count", "avg_selling_price"]] = (
    articles_clean[["product_purchase_count", "unique_customer_count", "avg_selling_price"]].fillna(0)
)


In [22]:
# Create customer-product modeling table
customer_product_features = (
    transactions_sample
    .merge(customers_clean, on="customer_id", how="left")
    .merge(articles_clean, on="article_id", how="left")
)

#### Save outputs


In [ ]:
articles_clean.to_csv(os.path.join(PROCESSED_DIR, "articles.csv"), index=False)
customers_clean.to_csv(os.path.join(PROCESSED_DIR, "customers.csv"), index=False)
transactions_sample.to_csv(os.path.join(PROCESSED_DIR, "transactions.csv"), index=False)
customer_product_features.to_csv(os.path.join(PROCESSED_DIR, "customer_product_modeling_sample.csv"), index=False)

In [ ]:
# Schema check
print("articles_clean columns:")
print(articles_clean.columns.tolist())

print("\ncustomers_clean columns:")
print(customers_clean.columns.tolist())

print("\ntransactions_sample columns:")
print(transactions_sample.columns.tolist())

articles_clean columns:
['article_id', 'product_code', 'product_name', 'product_type_no', 'product_type', 'product_group', 'graphical_appearance_no', 'graphical_appearance', 'colour_group_code', 'color_group', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_color', 'department_no', 'department', 'index_code', 'index', 'index_group_no', 'index_group', 'section_no', 'section', 'garment_group_no', 'garment_group', 'description', 'image_path', 'image_exists', 'product_purchase_count', 'unique_customer_count', 'avg_selling_price']

customers_clean columns:
['customer_id', 'fashion_news_binary', 'is_active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code', 'age_group']

transactions_sample columns:
['transaction_date', 'customer_id', 'article_id', 'price', 'sales_channel_id']
